# VQE Algorithm

Variational Quantum Eigensolvers (VQE) are a leading algorithm for noisy intermediate-scale quantum (NISQ) devices. They combine a parameterized quantum circuit (the *ansatz*) with a classical optimizer to approximate low-lying eigenvalues of a Hamiltonian by minimizing an energy expectation value.

In this notebook, we apply VQE to a two-spin (two-qubit) **Heisenberg XXZ** model and compare the variational estimate of the ground-state energy against an exact classical result.

## Problem Context

The (nearest-neighbor) Heisenberg model describes interactions between **spin-$\tfrac{1}{2}$** degrees of freedom on a lattice. For two sites, we map each spin-$\tfrac{1}{2}$ to a single qubit, so the Hilbert space is $\mathbb{C}^2 \otimes \mathbb{C}^2$.

> **Pauli operators and measurement.** The spin operators are proportional to the Pauli matrices: $S_\alpha = \tfrac{1}{2}\sigma_\alpha$ for $\alpha\in\{x,y,z\}$. Measuring a qubit in the $X$, $Y$, or $Z$ basis corresponds to measuring the observables $\sigma_x$, $\sigma_y$, or $\sigma_z$ (eigenvalues $\pm 1$).

> In VQE we estimate expectation values like $\langle \sigma_x\otimes\sigma_x \rangle$, $\langle \sigma_y\otimes\sigma_y \rangle$, and $\langle \sigma_z\otimes\sigma_z \rangle$ by running the circuit many times and averaging measurement outcomes in the appropriate bases.

### The Hamiltonian we’ll solve
A common two-qubit **XXZ** Heisenberg Hamiltonian is

$$H = J\,(\sigma_x\otimes\sigma_x + \sigma_y\otimes\sigma_y + \Delta\,\sigma_z\otimes\sigma_z),$$

where $J$ sets the interaction strength and $\Delta$ is the anisotropy. (Some conventions use $S_\alpha=\tfrac12\sigma_\alpha$, which changes the overall prefactor by $\tfrac14$; we’ll be explicit in code about which convention we use.)

VQE prepares a trial state $|\psi(\theta)\rangle$ and minimizes the energy

$$E(\theta)=\langle\psi(\theta)|H|\psi(\theta)\rangle,$$

estimated from measurement statistics.

This model is small enough to be solved exactly on a classical computer, allowing direct validation of the VQE results. At the same time, it captures key challenges of variational quantum algorithms, such as ansatz design, optimization landscapes, and measurement noise.

## Import Libraries

In [ ]:
# 1) Imports
import numpy as np
from numpy import pi

# Qiskit (core + primitives)
from qiskit import QuantumCircuit
from qiskit.quantum_info import SparsePauliOp
from qiskit.primitives import StatevectorEstimator
from qiskit.circuit import ParameterVector

# Plotting (optional later)
import matplotlib.pyplot as plt

## Ansatz (parameterized trial state)

The **ansatz** is the parameterized quantum circuit that prepares the trial state $|\psi(\theta)\rangle$. Instead of searching over all possible quantum states (exponentially large), VQE restricts the search to a *family* of states generated by varying the circuit parameters $\theta = (\theta_1, \theta_2, \dots, \theta_n)$.

### Design considerations
- **Expressivity**: Can the ansatz represent (or approximate) the true ground state? If your ansatz is too restrictive, VQE cannot reach the optimal energy.
- **Depth vs. trainability**: Deeper circuits are more expressive but harder to optimize and more susceptible to noise on NISQ hardware.
- **Problem structure**: Good ansätze exploit symmetries or physical intuition (e.g., particle conservation, spin sectors).

### Hardware-Efficient Ansatz (HEA)
We use a simple **hardware-efficient ansatz** with the following structure (repeated for `depth` layers):

1. **Local single-qubit rotations**: $R_y(\theta)$ and $R_z(\theta)$ on each qubit (together they span all single-qubit unitaries)
2. **Entangling layer**: CNOT gates to create correlations between qubits

For **depth=1**, we get **4 parameters** and a single entangling layer—enough flexibility to explore interesting entangled states while remaining shallow enough for near-term devices.

This class of ansätze is called "hardware-efficient" because it uses native gates available on most quantum processors (superconducting qubits, trapped ions, etc.), minimizing compilation overhead.

In [ ]:
# 2) Ansatz (parameterized trial state)
def he_ansatz_2q(depth: int = 1):
    """Hardware-efficient ansatz for 2 qubits.

    Structure per layer:
      - Single-qubit rotations: RY then RZ on each qubit
      - Entangler: CX(0 -> 1)

    Returns:
      circuit: parameterized QuantumCircuit
      theta: ParameterVector containing all parameters (length = 4*depth)
    """
    if depth < 1:
        raise ValueError("depth must be >= 1")

    theta = ParameterVector("θ", length=4 * depth)
    qc = QuantumCircuit(2, name=f"HEA2q_d{depth}")

    k = 0
    for _ in range(depth):
        # Local rotations
        qc.ry(theta[k + 0], 0)
        qc.rz(theta[k + 1], 0)
        qc.ry(theta[k + 2], 1)
        qc.rz(theta[k + 3], 1)
        k += 4
        # Entangling layer
        qc.cx(0, 1)

    return qc, theta

## Hamiltonian (the energy operator)

The **Hamiltonian** $H$ is the quantum operator that encodes the energy of the system. In VQE, the Hamiltonian defines the **objective function**: for any quantum state $|\psi\rangle$, the expectation value

$$
E = \langle\psi|H|\psi\rangle
$$

gives the energy of that state. The goal is to find the state that minimizes this energy—the **ground state**.

### Role in VQE
- The **ansatz** prepares trial states $|\psi(\theta)\rangle$ by varying parameters $\theta$
- The **Hamiltonian** scores each state by computing $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$
- The **classical optimizer** adjusts $\theta$ to minimize $E(\theta)$

By the **variational principle**, $E(\theta) \geq E_0$ (the true ground-state energy), so minimizing over $\theta$ gives the best approximation within the ansatz's expressivity.

### XXZ Heisenberg Hamiltonian structure
Our two-qubit Hamiltonian is

$$
H = J(\sigma_x\otimes\sigma_x + \sigma_y\otimes\sigma_y + \Delta\,\sigma_z\otimes\sigma_z),
$$

which is a sum of **Pauli strings** (products of Pauli operators acting on different qubits):
- Each term like $\sigma_x\otimes\sigma_x$ measures spin–spin correlation along one axis
- $J$ controls the overall interaction strength
- $\Delta$ is the **anisotropy parameter**: when $\Delta = 1$, the model is isotropic (rotationally symmetric); $\Delta \neq 1$ breaks symmetry along the $z$ axis

### Physical interpretation
- **$J > 0$ (antiferromagnetic)**: favors anti-aligned spins → singlet ground state
- **$J < 0$ (ferromagnetic)**: favors aligned spins → triplet ground state
- **$\Delta$**: tunes whether the system prefers alignment in the $xy$ plane ($\Delta < 1$) or along $z$ ($\Delta > 1$)

### Measurement on quantum hardware
To evaluate $\langle H\rangle$, we measure each Pauli term separately:
1. Prepare $|\psi(\theta)\rangle$ using the ansatz
2. Rotate qubits into the appropriate measurement basis ($X$, $Y$, or $Z$)
3. Measure many times and average the outcomes
4. Combine weighted averages: $\langle H\rangle = J\langle XX\rangle + J\langle YY\rangle + J\Delta\langle ZZ\rangle$

In this notebook, we use `StatevectorEstimator` for exact (noiseless) evaluation, but on real hardware you'd run circuits with finite shots.

In [ ]:
# 3) XXZ Heisenberg Hamiltonian
def heisenberg_xxz_2q(J: float = 1.0, delta: float = 1.0) -> SparsePauliOp:
    """Return H = J(XX + YY + delta ZZ) as a SparsePauliOp on 2 qubits."""
    paulis = ["XX", "YY", "ZZ"]
    coeffs = [J, J, J * delta]
    return SparsePauliOp(paulis, coeffs=np.asarray(coeffs, dtype=complex))

# Example Hamiltonian (you can change J and delta later)
J = 1.0
Delta = 1.0
H = heisenberg_xxz_2q(J=J, delta=Delta)
H

## Energy Evaluation

Now we connect the ansatz and Hamiltonian to compute the core quantity in VQE: the **energy expectation value** $E(\theta) = \langle\psi(\theta)|H|\psi(\theta)\rangle$.

### The evaluation workflow
1. **Bind parameters**: Take the abstract parameterized circuit (ansatz) and substitute concrete values for $\theta$ to get a specific quantum state $|\psi(\theta)\rangle$
2. **Run measurement**: Use a quantum estimator to evaluate $\langle\psi(\theta)|H|\psi(\theta)\rangle$
   - On real hardware: run the circuit many times with basis rotations for each Pauli term, average measurement outcomes
   - With `StatevectorEstimator`: compute the exact expectation value from the full statevector (no noise, no sampling error)
3. **Return energy**: The scalar result $E(\theta)$ is what the classical optimizer will try to minimize

### Why `StatevectorEstimator`?
For this tutorial, we use **exact statevector simulation** rather than shot-based sampling:
- ✅ **Validation**: eliminates measurement noise, letting us verify VQE's optimization performance cleanly
- ✅ **Speed**: for 2 qubits, exact simulation is faster than shot-based sampling
- ⚠️ **Not scalable**: statevector simulation scales exponentially ($2^n$ amplitudes for $n$ qubits), so it only works for small systems

When moving to larger problems or real quantum hardware, you'd switch to `Sampler` or `Estimator` primitives with finite shots.

### The `energy_expectation` function
This helper wraps the full evaluation pipeline:
```python
energy_expectation(circuit, hamiltonian, param_values) -> float
```
It's the bridge between the quantum ansatz and the classical optimizer—given any $\theta$, it returns the corresponding energy $E(\theta)$.

In [ ]:
# 4) Circuit + energy evaluation helper
estimator = StatevectorEstimator()

def energy_expectation(circuit: QuantumCircuit, hamiltonian: SparsePauliOp, param_values: np.ndarray) -> float:
    """Compute <psi(theta)|H|psi(theta)> using an exact statevector estimator."""
    bound = circuit.assign_parameters(param_values, inplace=False)
    job = estimator.run([(bound, hamiltonian)])
    result = job.result()[0]
    # result.data.evs is an array-like; for a single observable take first element
    return float(np.real(result.data.evs[0]))

# Build a parameterized circuit instance
ansatz, theta = he_ansatz_2q(depth=1)
ansatz.draw(output="text")